In [1]:
"""
================================================================================
otq_comm_stream_regression.py
================================================================================

SAVED LOCATION (IMPORTANT):
  H:\\My Drive\\Python\\Practice\\otq_comm_stream_regression.py

WHAT THIS MODULE DOES:
- Interactive, single-file OTQ communication analytics prototype
- Guides the user WHILE RUNNING to place datasets in the correct locations
- Ingests Slack / Teams / Outlook exports (CSV)
- Optionally ingests your dissertation (PDF/DOCX/TXT) to build a trust lexicon
- Builds daily features + a proxy outcome (response time)
- Runs regression and outputs a regression plot

RUN COMMAND (PowerShell):
  cd "H:\\My Drive\\Python\\Practice"
  python otq_comm_stream_regression.py

OPTIONAL NON-INTERACTIVE RUN:
  python otq_comm_stream_regression.py ^
    --inputs ".\\data\\raw\\comm_logs" ^
    --dissertation ".\\data\\raw\\OTQ_dissertation.pdf" ^
    --out ".\\reports\\figures" ^
    --non_interactive

FOLDER STRUCTURE (RECOMMENDED):
  Practice\
  ├─ otq_comm_stream_regression.py        <-- THIS FILE
  ├─ data\
  │  └─ raw\
  │     ├─ comm_logs\
  │     │  ├─ slack_messages.csv
  │     │  ├─ teams_meetings.csv
  │     │  └─ outlook_calendar.csv
  │     └─ OTQ_dissertation.pdf
  └─ reports\
     └─ figures\

================================================================================
"""

# ==============================================================================
# IMPORTS
# ==============================================================================
import argparse
import math
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


# ==============================================================================
# BASIC HELPERS
# ==============================================================================
def safe_int(x, default=0) -> int:
    try:
        return int(x)
    except Exception:
        return default


def safe_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def parse_datetime(value: Any) -> Optional[pd.Timestamp]:
    """
    Handles:
    - ISO timestamps
    - Outlook / Teams CSV timestamps
    - Slack epoch timestamps (e.g., 1700000000.123)
    """
    if value is None:
        return None

    s = str(value).strip()
    if not s:
        return None

    # Slack epoch seconds
    if re.fullmatch(r"\d{10}(\.\d+)?", s):
        return pd.to_datetime(float(s), unit="s", utc=True)

    ts = pd.to_datetime(s, errors="coerce", utc=True)
    return None if ts is pd.NaT else ts


# ==============================================================================
# NEW (INTERACTIVE): User-facing folder guidance + file checks
# ------------------------------------------------------------------------------
# CHANGE NOTE:
# - Added interactive helpers that:
#   * print the expected folder tree
#   * create missing folders (optional)
#   * repeatedly check for required dataset files
#   * ask the user to place datasets in the correct location, then press Enter
# ==============================================================================
def print_recommended_tree(base: Path) -> None:
    print("\n=== Recommended Folder Layout (Practice root) ===")
    print(f"{base}\\")
    print("├─ otq_comm_stream_regression.py")
    print("├─ data\\")
    print("│  ├─ raw\\")
    print("│  │  ├─ comm_logs\\")
    print("│  │  │  ├─ slack_messages.csv")
    print("│  │  │  ├─ teams_meetings.csv")
    print("│  │  │  └─ outlook_calendar.csv")
    print("│  │  └─ OTQ_dissertation.pdf   (optional)")
    print("│  └─ processed\\   (optional)")
    print("└─ reports\\")
    print("   └─ figures\\")


def prompt_yes_no(question: str, default_yes: bool = True) -> bool:
    """
    Small interactive helper. Example:
      prompt_yes_no("Create folders now?", default_yes=True)
    """
    default = "Y/n" if default_yes else "y/N"
    while True:
        ans = input(f"{question} [{default}]: ").strip().lower()
        if not ans:
            return default_yes
        if ans in ("y", "yes"):
            return True
        if ans in ("n", "no"):
            return False
        print("Please type y or n.")


def ensure_project_folders(base: Path, inputs_dir: Path, out_dir: Path, non_interactive: bool) -> None:
    """
    CHANGE NOTE:
    - Creates folders if missing, with user confirmation (unless non_interactive).
    """
    needed = [
        inputs_dir,
        base / "data" / "raw",
        base / "data" / "processed",
        out_dir,
    ]

    missing = [p for p in needed if not p.exists()]
    if not missing:
        return

    print("\nI don't see some expected folders yet:")
    for p in missing:
        print(f" - {p}")

    if non_interactive:
        for p in missing:
            ensure_dir(p)
        print("Created missing folders (non-interactive mode).")
        return

    if prompt_yes_no("Do you want me to create these folders now?", default_yes=True):
        for p in missing:
            ensure_dir(p)
        print("Folders created.")
    else:
        print("Okay — please create the folders yourself, then re-run.")
        sys.exit(1)


def wait_for_files(required_files: List[Path], non_interactive: bool) -> None:
    """
    CHANGE NOTE:
    - Repeatedly checks for required files. In interactive mode:
      * tells user exactly where to put them
      * pauses until user presses Enter (then re-checks)
    """
    def missing_files() -> List[Path]:
        return [p for p in required_files if not p.exists()]

    miss = missing_files()
    if not miss:
        return

    print("\n=== Dataset Placement Check ===")
    print("I can't find the following required dataset file(s):")
    for p in miss:
        print(f" - {p}")

    if non_interactive:
        print("\nNon-interactive mode: exiting because required files are missing.")
        print("Place the files in the paths above, then re-run.")
        sys.exit(1)

    print("\nPlease do this now:")
    print("1) Export or create the datasets")
    print("2) Save them EXACTLY to the file paths listed above")
    print("3) Come back here and press Enter so I can re-check\n")

    while True:
        input("Press Enter after you have placed the files...")
        miss = missing_files()
        if not miss:
            print("✅ All required files found. Continuing...\n")
            return
        print("\nStill missing:")
        for p in miss:
            print(f" - {p}")
        print("Please place them and press Enter again.\n")


def optional_wait_for_file(optional_file: Path, label: str, non_interactive: bool) -> bool:
    """
    CHANGE NOTE:
    - For optional dissertation file:
      * asks user if they want to use it
      * if yes, pauses until file is present
    Returns True if file will be used.
    """
    if non_interactive:
        return optional_file.exists()

    if optional_file.exists():
        return True

    use_it = prompt_yes_no(
        f"Optional: Do you want to include your {label} to build the OTQ lexicon?",
        default_yes=True,
    )
    if not use_it:
        return False

    print(f"\nOkay — please place your {label} here:")
    print(f" - {optional_file}")
    print("Then press Enter so I can re-check.\n")

    while True:
        input("Press Enter after placing the file...")
        if optional_file.exists():
            print(f"✅ Found {label}. Continuing with dissertation lexicon.\n")
            return True
        print(f"Still not found: {optional_file}\n")


# ==============================================================================
# DISSERTATION INGESTION + OTQ LEXICON
# ==============================================================================
STOPWORDS = {
    "the", "and", "a", "to", "of", "in", "for", "on", "with", "as", "by", "is", "are", "was",
    "were", "be", "been", "being", "that", "this", "it", "from", "or", "an", "at", "we",
    "you", "they", "i", "our", "your", "their"
}


def tokenize(text: str) -> List[str]:
    tokens = re.findall(r"[a-z]{3,}", (text or "").lower())
    return [t for t in tokens if t not in STOPWORDS]


def load_dissertation_text(path: Optional[Path]) -> str:
    """
    Supports: PDF / DOCX / TXT
    """
    if not path or not path.exists():
        return ""

    suffix = path.suffix.lower()

    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="ignore")

    if suffix == ".docx":
        # Requires python-docx
        import docx
        doc = docx.Document(str(path))
        return "\n".join(p.text for p in doc.paragraphs)

    if suffix == ".pdf":
        # Requires pypdf
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    return ""


def build_otq_lexicon(text: str, top_n: int = 200) -> Dict[str, float]:
    tokens = tokenize(text)
    if not tokens:
        return {}

    counts = Counter(tokens)
    max_freq = max(counts.values())
    return {word: freq / max_freq for word, freq in counts.most_common(top_n)}


def trust_features(text: str, lexicon: Dict[str, float]) -> Tuple[int, float]:
    tokens = tokenize(text)
    if not tokens:
        return 0, 0.0
    hits = sum(1 for t in tokens if t in lexicon)
    density = hits / len(tokens) * 100
    return hits, density


# ==============================================================================
# OTQ EVENT SCORING
# ==============================================================================
def otq_event_units(event_type: str, duration_min: float, participant_count: int) -> float:
    t = (event_type or "").lower()

    if "meeting" in t:
        base = 24.5
    elif "chat" in t or "slack" in t:
        base = 9.5
    elif "email" in t:
        base = 6.0
    else:
        base = 5.0

    dur_bonus = min(max(duration_min, 0.0), 120.0) / 60.0
    part_bonus = math.log(max(participant_count, 1), 2) * 1.5
    return float(base + dur_bonus + part_bonus)


# ==============================================================================
# INPUT PARSERS (CSV)
# ------------------------------------------------------------------------------
# CHANGE NOTE:
# - Kept parsers simple and explicit for your starter datasets.
# - If columns differ, you can adjust column names here.
# ==============================================================================
def load_csv_any(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def parse_slack_csv(path: Path) -> pd.DataFrame:
    """
    Expected columns:
      ts,user,channel,text

    ts can be ISO datetime OR Slack epoch seconds.
    """
    df = load_csv_any(path)
    # Defensive: allow common alternate column names
    col_ts = "ts" if "ts" in df.columns else ("timestamp" if "timestamp" in df.columns else None)
    col_text = "text" if "text" in df.columns else ("message" if "message" in df.columns else None)
    col_channel = "channel" if "channel" in df.columns else ("room" if "room" in df.columns else None)

    if not col_ts or not col_text:
        raise ValueError(f"Slack CSV missing required columns. Need ts/text. Found: {list(df.columns)}")

    return pd.DataFrame({
        "source": "slack",
        "event_type": "chat",
        "timestamp": df[col_ts],
        "text": df[col_text],
        "duration_min": 0.0,
        "participant_count": 2,  # proxy
        "channel": df[col_channel] if col_channel else "slack",
    })


def parse_teams_csv(path: Path) -> pd.DataFrame:
    """
    Expected columns:
      timestamp,subject,duration,participant_count,organizer  (organizer optional)

    timestamp should be ISO datetime.
    duration is minutes.
    """
    df = load_csv_any(path)
    required = ["timestamp", "subject", "duration", "participant_count"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Teams CSV missing columns {missing}. Found: {list(df.columns)}")

    return pd.DataFrame({
        "source": "teams",
        "event_type": "meeting",
        "timestamp": df["timestamp"],
        "text": df["subject"],
        "duration_min": df["duration"],
        "participant_count": df["participant_count"],
        "channel": "teams",
    })


def parse_outlook_csv(path: Path) -> pd.DataFrame:
    """
    Expected columns:
      Subject,Start,End

    Start/End should be parseable datetimes.
    """
    df = load_csv_any(path)
    required = ["Subject", "Start", "End"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Outlook CSV missing columns {missing}. Found: {list(df.columns)}")

    # Duration derived from Start/End
    start_dt = pd.to_datetime(df["Start"], errors="coerce", utc=True)
    end_dt = pd.to_datetime(df["End"], errors="coerce", utc=True)
    duration = (end_dt - start_dt).dt.total_seconds() / 60.0

    return pd.DataFrame({
        "source": "outlook",
        "event_type": "meeting",
        "timestamp": df["Start"],
        "text": df["Subject"],
        "duration_min": duration.fillna(0.0).astype(float),
        "participant_count": 3,  # proxy unless you add attendee parsing
        "channel": "calendar",
    })


# ==============================================================================
# EVENT TABLE + FEATURE ENGINEERING
# ==============================================================================
def build_event_table(dfs: List[pd.DataFrame], lexicon: Dict[str, float]) -> pd.DataFrame:
    df = pd.concat(dfs, ignore_index=True)

    df["ts"] = df["timestamp"].apply(parse_datetime)
    df = df.dropna(subset=["ts"]).copy()
    df["date"] = df["ts"].dt.floor("D")

    df["duration_min"] = df["duration_min"].apply(safe_float)
    df["participant_count"] = df["participant_count"].apply(safe_int)

    df["text"] = df["text"].fillna("").astype(str)
    df["text_len"] = df["text"].str.len()

    df["eu"] = df.apply(
        lambda r: otq_event_units(r["event_type"], r["duration_min"], r["participant_count"]),
        axis=1
    )

    # Dissertation-based trust signals
    trust = df["text"].apply(lambda t: trust_features(t, lexicon) if lexicon else (0, 0.0))
    df["trust_hits"] = trust.apply(lambda x: x[0])
    df["trust_density"] = trust.apply(lambda x: x[1])

    return df


def daily_features(events: pd.DataFrame) -> pd.DataFrame:
    d = events.groupby("date", as_index=False).agg(
        total_eu=("eu", "sum"),
        total_events=("eu", "count"),
        avg_participants=("participant_count", "mean"),
        avg_duration=("duration_min", "mean"),
        total_text=("text_len", "sum"),
        trust_hits=("trust_hits", "sum"),
        trust_density=("trust_density", "mean"),
    )

    # Light transforms help regression
    d["log_total_eu"] = np.log1p(d["total_eu"])
    d["log_total_events"] = np.log1p(d["total_events"])
    d["log_total_text"] = np.log1p(d["total_text"].clip(lower=0))
    d["log_trust_hits"] = np.log1p(d["trust_hits"].clip(lower=0))
    return d


# ==============================================================================
# OUTCOME PROXY: Response Time (minutes)
# ------------------------------------------------------------------------------
# CHANGE NOTE:
# - Uses time between consecutive events in the same source+channel.
# - This is a proxy outcome; replace later with real KPI (SPI/CPI, tickets closed, etc.)
# ==============================================================================
def compute_outcome_proxy(events: pd.DataFrame) -> pd.DataFrame:
    e = events.sort_values("ts").copy()
    e["channel_key"] = e["source"].astype(str) + "::" + e["channel"].astype(str)
    e["prev_ts"] = e.groupby("channel_key")["ts"].shift(1)
    e["delta_min"] = (e["ts"] - e["prev_ts"]).dt.total_seconds() / 60.0

    # Keep reasonable gaps: 0 < delta <= 12 hours
    e = e[(e["delta_min"].notna()) & (e["delta_min"] > 0) & (e["delta_min"] <= 720)].copy()

    out = e.groupby("date", as_index=False).agg(
        median_response_min=("delta_min", "median"),
        samples=("delta_min", "count"),
    )
    return out


# ==============================================================================
# REGRESSION + PLOTS
# ==============================================================================
def run_regression(features: pd.DataFrame, outcome: pd.DataFrame, out_dir: Path) -> None:
    df = features.merge(outcome, on="date", how="inner").copy()

    if len(df) < 10:
        print("\n[ERROR] Not enough merged daily rows for regression (need ~10+).")
        print("Add more days of communication logs (spread timestamps across days), then re-run.")
        return

    feature_cols = [
        "log_total_eu",
        "log_total_events",
        "avg_participants",
        "avg_duration",
        "log_total_text",
        "log_trust_hits",
        "trust_density",
    ]

    X = df[feature_cols].fillna(0.0).to_numpy()
    y = df["median_response_min"].fillna(df["median_response_min"].median()).to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("\n=== Regression Results ===")
    print(f"Test R^2: {r2_score(y_test, preds):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, preds):.2f} minutes")

    ensure_dir(out_dir)

    # Plot: Actual vs Predicted
    plt.figure()
    plt.scatter(y_test, preds)
    plt.xlabel("Actual median response (min)")
    plt.ylabel("Predicted median response (min)")
    plt.title("OTQ Regression: Actual vs Predicted")
    fig_path = out_dir / "actual_vs_predicted.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()

    # Save modeling table
    table_path = out_dir / "daily_model_table.csv"
    df.to_csv(table_path, index=False)

    print("\nSaved outputs:")
    print(f" - {fig_path}")
    print(f" - {table_path}")


# ==============================================================================
# MAIN (Interactive workflow)
# ------------------------------------------------------------------------------
# CHANGE NOTE:
# - If you run with no args, it defaults to your Practice\ structure and
#   guides you to place datasets in the right spot.
# - It pauses and re-checks until files are present (interactive mode).
# ==============================================================================
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--inputs",
        type=str,
        default="",
        help="Folder containing comm logs (default: .\\data\\raw\\comm_logs)",
    )
    parser.add_argument(
        "--dissertation",
        type=str,
        default="",
        help="Path to dissertation (default: .\\data\\raw\\OTQ_dissertation.pdf)",
    )
    parser.add_argument(
        "--out",
        type=str,
        default="",
        help="Output folder for plots (default: .\\reports\\figures)",
    )
    parser.add_argument(
        "--lexicon_top_n",
        type=int,
        default=200,
        help="How many dissertation-derived keywords to keep in the lexicon.",
    )
    parser.add_argument(
        "--non_interactive",
        action="store_true",
        help="Disable prompts; exit if required files are missing.",
    )
    # CHANGE NOTE:
# Jupyter/IPython injects extra arguments like:
#   -f C:\\Users\\...\\kernel-xxxx.json
# parse_known_args() allows us to ignore them safely.
    args, _unknown = parser.parse_known_args()

    if _unknown:
        print(f"[INFO] Ignored unknown arguments: {_unknown}")


    base = Path.cwd().resolve()

    # Defaults aligned to your Practice\ structure
    inputs_dir = Path(args.inputs).resolve() if args.inputs else (base / "data" / "raw" / "comm_logs")
    out_dir = Path(args.out).resolve() if args.out else (base / "reports" / "figures")
    dissertation_path = (
        Path(args.dissertation).resolve()
        if args.dissertation
        else (base / "data" / "raw" / "OTQ_dissertation.pdf")
    )

    print("\n================================================================================")
    print("OTQ Communication Regression (Interactive)")
    print("================================================================================")
    print(f"Running from: {base}")
    print_recommended_tree(base)

    # Ensure folders exist (prompt user unless non_interactive)
    ensure_project_folders(base, inputs_dir, out_dir, non_interactive=args.non_interactive)

    # Required datasets
    slack_file = inputs_dir / "slack_messages.csv"
    teams_file = inputs_dir / "teams_meetings.csv"
    outlook_file = inputs_dir / "outlook_calendar.csv"

    # CHANGE NOTE:
    # - Ask user to place datasets while running (interactive loop)
    required_files = [slack_file, teams_file, outlook_file]
    wait_for_files(required_files, non_interactive=args.non_interactive)

    # Optional dissertation usage
    use_dissertation = optional_wait_for_file(
        dissertation_path,
        label="dissertation file (PDF/DOCX/TXT)",
        non_interactive=args.non_interactive
    )

    # Build lexicon
    lexicon: Dict[str, float] = {}
    if use_dissertation:
        try:
            dis_text = load_dissertation_text(dissertation_path)
            lexicon = build_otq_lexicon(dis_text, top_n=args.lexicon_top_n)
            print(f"Loaded dissertation lexicon terms: {len(lexicon)}")
        except Exception as e:
            print(f"[WARN] Could not load dissertation/lexicon: {e}")
            print("Continuing WITHOUT dissertation-based trust features.")
            lexicon = {}

    # Parse data
    try:
        slack_df = parse_slack_csv(slack_file)
        teams_df = parse_teams_csv(teams_file)
        outlook_df = parse_outlook_csv(outlook_file)
    except Exception as e:
        print("\n[ERROR] Failed parsing one of the datasets.")
        print("Details:", e)
        print("\nTip: Check column names match expected formats in the parser functions.")
        sys.exit(1)

    # Build events/features/outcome
    events = build_event_table([slack_df, teams_df, outlook_df], lexicon=lexicon)
    daily = daily_features(events)
    outcome = compute_outcome_proxy(events)

    print(f"\nLoaded events: {len(events):,}")
    print(f"Daily rows:    {len(daily):,}")
    print(f"Outcome rows:  {len(outcome):,}")

    # Run regression + save plot
    run_regression(daily, outcome, out_dir)


if __name__ == "__main__":
    main()


[INFO] Ignored unknown arguments: ['-f', 'C:\\Users\\vcox\\AppData\\Roaming\\jupyter\\runtime\\kernel-26279007-6f7a-4633-8f7f-faa7fafcaa7e.json']

OTQ Communication Regression (Interactive)
Running from: H:\My Drive\Python\Practice

=== Recommended Folder Layout (Practice root) ===
H:\My Drive\Python\Practice\
├─ otq_comm_stream_regression.py
├─ data\
│  ├─ raw\
│  │  ├─ comm_logs\
│  │  │  ├─ slack_messages.csv
│  │  │  ├─ teams_meetings.csv
│  │  │  └─ outlook_calendar.csv
│  │  └─ OTQ_dissertation.pdf   (optional)
│  └─ processed\   (optional)
└─ reports\
   └─ figures\


Optional: Do you want to include your dissertation file (PDF/DOCX/TXT) to build the OTQ lexicon? [Y/n]:  n



[ERROR] Failed parsing one of the datasets.
Details: Teams CSV missing columns ['timestamp', 'subject', 'duration', 'participant_count']. Found: ['ts', 'user', 'channel', 'text']

Tip: Check column names match expected formats in the parser functions.


SystemExit: 1

C:\Users\vcox\.conda\envs\ds\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
